In [1]:
import sys
sys.path.append("../")
from chess_engine.src.model.classes.sqlite.models import GamePositionRollup
import numpy as np
from tqdm import tqdm
from chess_engine.src.model.classes.bitboard_processing.bitboard_creator import bitboards_to_array, sample_bitboard_dict
from chess_engine.src.model.classes.sqlite.database import  get_db
from chess_engine.src.model.config.config import data_settings
import os
import h5py
import glob
import torch
from torch.utils.data import Dataset, DataLoader

In [2]:
def delete_all_files(directory):
    # Check if the directory exists
    if not os.path.exists(directory):
        print(f"The directory {directory} does not exist.")
        return
    
    # Iterate over all files in the directory
    for filename in os.listdir(directory):
        file_path = os.path.join(directory, filename)
        try:
            # Check if it's a file (not a subdirectory)
            if os.path.isfile(file_path):
                os.remove(file_path)  # Delete the file
                print(f"Deleted: {file_path}")
        except Exception as e:
            print(f"Failed to delete {file_path}. Reason: {e}")


In [3]:
data_settings.BatchFileSize

1000000

In [11]:
def db_to_hdf5_files():
    batch_file_size = data_settings.BatchFileSize  # Number of examples per file
    
    sets = {
        data_settings.TrainingDirectory: GamePositionRollup.is_training_data.is_(True),
        data_settings.TestingDirectory: GamePositionRollup.is_testing_data.is_(True),
        data_settings.ValidationDirectory: GamePositionRollup.is_validation_data.is_(True)
    }

    for h5_dir, filter_conditions in sets.items():
        
        delete_all_files(h5_dir)
        idx = 0
        with next(get_db()) as session:
            # Count total number of records
            total_records = session.query(GamePositionRollup).filter(
                filter_conditions
            ).count()
            print(f"Total records: {total_records}")

            # Process records in chunks
            for batch_start in range(0, total_records, batch_file_size):
                features_list = []
                labels_list = []

                # Fetch batch of records
                records = session.query(GamePositionRollup).filter(
                    filter_conditions
                ).offset(batch_start).limit(batch_file_size).all()

                for record in records:
                    # Extract features
                    features = [getattr(record, attr) for attr in sample_bitboard_dict.keys()]
                    features = bitboards_to_array(features)  # Convert to array format

                    # Extract labels
                    labels = record.win_buckets

                    # Append to batch
                    features_list.append(features)
                    labels_list.append(labels)

                # Convert lists to NumPy arrays
                features_array = np.array(features_list)  # Shape: (batch_file_size, 12, 8, 8)
                labels_array = np.array(labels_list)      # Shape: (batch_file_size, 3)

                # Save to an HDF5 file
                batch_file_name = os.path.join(h5_dir, f'data_{idx}.h5')
                with h5py.File(batch_file_name, 'w') as f:
                    # Create datasets with compression
                    f.create_dataset('features', data=features_array, compression='gzip')
                    f.create_dataset('labels', data=labels_array, compression='gzip')

                print(f"Saved batch {idx} to {batch_file_name}")

                idx += 1

In [ ]:
db_to_hdf5_files()

In [4]:


class HDF5Dataset(Dataset):
    def __init__(self, directory, transform=None):
        super().__init__()
        self.directory = directory
        self.transform = transform
        
        # Collect all .h5 files
        raw_file_paths = sorted(glob.glob(os.path.join(directory, '*.h5')))
        if not raw_file_paths:
            raise FileNotFoundError(f"No .h5 files found in {directory}")

        # Normalize or make all paths absolute to avoid mismatch
        self.files = [os.path.abspath(os.path.normpath(f)) for f in raw_file_paths]

        self.file_index_ranges = []
        cumulative_samples = 0

        # Just read shape info for indexing
        for fpath in self.files:
            with h5py.File(fpath, 'r') as h5f:
                num_samples = h5f['features'].shape[0]  # shape: (m, 8, 8, 12)
            self.file_index_ranges.append((fpath, cumulative_samples, cumulative_samples + num_samples))
            cumulative_samples += num_samples

        self.total_samples = cumulative_samples
        print(f"total samples: {self.total_samples}")
        # Will hold open file handles once per worker
        self.h5_files = {}

    def open_files(self):
        """Open each file once per worker, store the handle."""
        for fpath, _, _ in self.file_index_ranges:
            # Use the normalized path here
            fpath_norm = os.path.abspath(os.path.normpath(fpath))
            self.h5_files[fpath_norm] = h5py.File(fpath_norm, 'r')

    def close_files(self):
        """Close all file handles."""
        for f in self.h5_files.values():
            f.close()
        self.h5_files = {}

    def __len__(self):
        return self.total_samples

    def _get_file_index(self, idx):
        # Find which file the global index belongs to
        for fpath, start, end in self.file_index_ranges:
            if start <= idx < end:
                local_idx = idx - start
                return os.path.abspath(os.path.normpath(fpath)), local_idx
        raise IndexError(f"Index {idx} out of range")

    def __getitem__(self, idx):
        # Which file + local index in that file
        # print(f"Getting items: {idx}")
        if not self.h5_files:
            # print("Tried opening files")
            self.open_files()
            
        
        fpath_norm, local_idx = self._get_file_index(idx)

        # Access the already-open file handle
        h5f = self.h5_files[fpath_norm]

        # Extract features, labels
        features = h5f['features'][local_idx]  # shape: (8, 8, 12)
        labels = h5f['labels'][local_idx]      # shape: (3,)

        # If your model expects (12, 8, 8):
        features = features.transpose((2, 0, 1))

        if self.transform:
            features, labels = self.transform(features, labels)

        features = torch.from_numpy(features).float()
        labels = torch.from_numpy(labels).float()
        return features, labels


In [6]:


def worker_init_fn(worker_id):
    worker_info = torch.utils.data.get_worker_info()
    if worker_info is not None:
        dataset = worker_info.dataset
        dataset.open_files()

def create_dataloader(directory, batch_size=64, num_workers=4, shuffle=True):
    dataset = HDF5Dataset(directory=directory)
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True,
        worker_init_fn=worker_init_fn
    )
    return loader



batch_size = 512

train_loader = create_dataloader(data_settings.TrainingDirectory, batch_size=8192, num_workers=0, shuffle=False)

i = 0
# Example of using the DataLoader in a training loop
for epoch in range(1, 6):
    print(f"Epoch {epoch}")
    for i, (features, labels) in enumerate(train_loader):
        i = i + 1
        print(len(train_loader))

        print(f"grabbed item: {i*batch_size}")
        # features: (batch_size, 12, 8, 8)
        # labels:   (batch_size, 3)
        # ... pass through model, compute loss, backprop, etc. ...
        pass


total samples: 14452
Epoch 1
2
grabbed item: 512
2
grabbed item: 1024
Epoch 2
2
grabbed item: 512
2
grabbed item: 1024
Epoch 3
2
grabbed item: 512
2
grabbed item: 1024
Epoch 4
2
grabbed item: 512
2
grabbed item: 1024
Epoch 5
2
grabbed item: 512
2
grabbed item: 1024
